### 팬크로 매틱 영상과 멀티스펙트럼 영상 결합

In [11]:
import numpy as np
import rasterio
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from rasterio.plot import reshape_as_raster, reshape_as_image
from rasterio.enums import Resampling
from rasterio.warp import reproject

# Step 1. XML 에서 방사보정계수와 태양 고도각 찾기
def extract_radiometric_factors(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # 태양 고도각(Sun Elevation) 찾기
    sun_elevation = float(root.find(".//MEANSUNEL").text)
    
    # 방사 보정 계수(ABSCALFACTOR) 찾기
    radiometric_factors = {}
    for band in root.findall(".//IMD/*"):
        if band.tag.startswith("BAND_"):
            band_name = band.tag
            abs_cal_factor = float(band.find("ABSCALFACTOR").text)
            radiometric_factors[band_name] = abs_cal_factor
    
    return sun_elevation, radiometric_factors

# Step 2. 방사보정 적용(멀티스펙트럼 & 팬 밴드)
def apply_radimetric_correction(dn_array, abscal_factor, sun_elevation):
    theta = np.deg2rad(90 - sun_elevation)
    cos_theta = max(np.cos(theta), 0.2) # 분모가 0이 되는 것 방지
    
    reflectance = (dn_array * abscal_factor) / cos_theta
    # reflectance = np.clip(reflectance, 0, 1) # 반사율은 0~1 사이로 클리핑
    return reflectance

# Step 3. TIF 데이터 로드 및 방사 보정 적용
def process_tif_with_correction(tif_file, xml_file):
    sun_elevation, radiometric_factors = extract_radiometric_factors(xml_file)
    
    with rasterio.open(tif_file) as src:
        bands_data = {}
        band_names = list(radiometric_factors.keys())
        
        for i, band_name in enumerate(band_names):
            band_array = src.read(i+1).astype(np.float32)
            abscal_factor = radiometric_factors[band_name]
            
            reflectance = apply_radimetric_correction(band_array, abscal_factor, sun_elevation)
            bands_data[band_name] = reflectance
    
    return bands_data , src.meta

# Step 4. 멀티스펙트럼을 팬크로 매틱 크기로 업샘플링 (GDAL 방식과 동일)
def upsample_multi_to_pan(multi_data, multi_meta, pan_meta):
    upsampled_data = {}

    for band_name, band_array in multi_data.items():
        upsampled_band = np.zeros((pan_meta["height"], pan_meta["width"]), dtype=np.float32)

        reproject(
            source=band_array,
            destination=upsampled_band,
            src_transform=multi_meta["transform"],
            dst_transform=pan_meta["transform"],
            src_crs=multi_meta["crs"],
            dst_crs=pan_meta["crs"],
            resampling=Resampling.cubic
        )

        upsampled_data[band_name] = upsampled_band  # ✅ 밴드 이름 유지
    
    return upsampled_data

def pansharpen_brovey(pan_band, multi_bands):
    # WorldView 데이터의 밴드 이름에 맞게 수정
    # 밴드 5: Red, 밴드 3: Green, 밴드 2: Blue

    # R, G, B 밴드만 추출하여 Brovey 변환에 사용
    # multi_bands 딕셔너리의 키를 확인하여 밴드명 수정 필요
    # 예: multi_bands["BAND_5"], multi_bands["BAND_3"], multi_bands["BAND_2"]

    rgb_bands = [multi_bands[band] for band in ["BAND_R", "BAND_G", "BAND_B"]]

    # RGB 밴드의 합산 (분모에 작은 값 더해 0으로 나누는 오류 방지)
    multi_sum = np.sum(rgb_bands, axis=0) + 1e-6

    # 각 밴드별 팬샤프닝 적용 (R, G, B 밴드에만 적용)
    pan_sharpened_bands = {}
    for band_name in ["BAND_R", "BAND_G", "BAND_B"]:
        pan_sharpened_bands[band_name] = multi_bands[band_name] * (pan_band / multi_sum)

    return pan_sharpened_bands

# Step 6. 팬샤프닝 결과 저장 (좌표 정보 유지)
def save_pansharpened_tif(output_tif, pan_sharpened, meta):
    meta.update(count=len(pan_sharpened), dtype=rasterio.float32)

    with rasterio.open(output_tif, 'w', **meta) as dst:
        for i, (band_name, band_array) in enumerate(pan_sharpened.items()):
            dst.write(band_array.astype(rasterio.float32), i + 1)
            
# Step 7. 식생지수 계산(NDVI, GNDVI)
def calculate_vegetation_indices(pan_sharpened):
    nir_band = pan_sharpened["BAND_N"]  # NIR 밴드 
    red_band = pan_sharpened["BAND_R"]  # RED 밴드
    green_band = pan_sharpened["BAND_G"]  # GREEN 밴드
    
    # NDVI 계산
    ndvi = (nir_band - red_band) / (nir_band + red_band + 1e-6)

    # GNDVI 계산
    gndvi = (nir_band - green_band) / (nir_band + green_band + 1e-6)

    return {"NDVI": ndvi, "GNDVI": gndvi}

# Step 8. RGB 이미지 생성 함수
def generate_rgb_image(bands_dict, output_path, meta):
    """
    R, G, B 밴드 데이터 딕셔너리를 입력받아 RGB GeoTIFF로 저장
    """
    # 딕셔너리에서 R, G, B 밴드 배열을 추출하여 스택 (순서에 유의)
    # WorldView-3 밴드 번호 기준: R=5, G=3, B=2
    rgb_array = np.stack([bands_dict["BAND_R"], bands_dict["BAND_G"], bands_dict["BAND_B"]], axis=0)

    # 기존 메타데이터를 기반으로 새로운 메타데이터 생성
    new_meta = meta.copy()
    new_meta.update(
        count=3,
        dtype=rgb_array.dtype,
        transform=meta["transform"],
        crs=meta["crs"],
        height=rgb_array.shape[1],
        width=rgb_array.shape[2]
    )

    with rasterio.open(output_path, 'w', **new_meta) as dst:
        for i in range(3):
            dst.write(rgb_array[i], i + 1)

    print(f"✅ RGB 이미지 저장 완료: {output_path}")

# Step 9. 방사 보정 후 RGB 이미지 생성 (멀티스펙트럼 & 팬크로매틱)
def process_and_save_rgb_images(multi_reflectance, pan_sharpened, multi_meta, pan_meta):
    """
    - 멀티스펙트럼 (방사 보정 후) → RGB 이미지 저장
    - 팬샤프닝 적용된 데이터 → RGB 이미지 저장
    """
    output_rgb_multi = "산출물/RGB_Multi_50cm.tif"
    output_rgb_pan = "산출물/RGB_Pansharpened_50cm.tif"

    # 📌 1. 멀티스펙트럼에서 R, G, B 추출 후 RGB 저장 (위치 정보 포함)
    generate_rgb_image(multi_reflectance, output_rgb_multi, multi_meta)

    # 📌 2. 팬샤프닝 적용한 R, G, B 추출 후 RGB 저장 (팬샤프닝된 데이터의 위치 정보 유지)
    generate_rgb_image(pan_sharpened, output_rgb_pan, pan_meta)

    print(f"✅ 멀티스펙트럼 RGB 저장 완료: {output_rgb_multi}")
    print(f"✅ 팬샤프닝 RGB 저장 완료: {output_rgb_pan}")  


In [12]:
multi_tif = "A지역/050301576010_01_pan, mul(8Band), 50cm/050301576010_01_P001_MUL/22SEP17022645-M2AS-050301576010_01_P001.TIF"
pan_tif = "A지역/050301576010_01_pan, mul(8Band), 50cm/050301576010_01_P001_MUL/22SEP17022645-M2AS-050301576010_01_P001.TIF"
multi_xml = "A지역/050301576010_01_pan, mul(8Band), 50cm/050301576010_01_P001_MUL/22SEP17022645-M2AS-050301576010_01_P001.XML"
pan_xml = "A지역/050301576010_01_pan, mul(8Band), 50cm/050301576010_01_P001_PAN/22SEP17022645-P2AS-050301576010_01_P001.XML"
output_tif = "산출물/Pansharpened_Output_50cm.tif"

# 1. 멀티 스펙트럼 방사 보정 적용
multi_reflectance, multi_meta = process_tif_with_correction(multi_tif, multi_xml)

# 2. 팬크로매틱 방사 보정 적용
pan_reflectance, pan_meta = process_tif_with_correction(pan_tif, pan_xml)

# 3. 멀티스펙트럼을 팬크로매틱 해상도로 업샘플링
multi_upsampled = upsample_multi_to_pan(multi_reflectance, multi_meta, pan_meta)

# 4. 팬샤프닝 적용 (Brovey Transform)
pan_sharpened = pansharpen_brovey(pan_reflectance["BAND_P"], multi_upsampled)

# 5. RGB 이미지 생성 및 저장
process_and_save_rgb_images(multi_reflectance, pan_sharpened, multi_meta, pan_meta)

✅ RGB 이미지 저장 완료: 산출물/RGB_Multi_50cm.tif
✅ RGB 이미지 저장 완료: 산출물/RGB_Pansharpened_50cm.tif
✅ 멀티스펙트럼 RGB 저장 완료: 산출물/RGB_Multi_50cm.tif
✅ 팬샤프닝 RGB 저장 완료: 산출물/RGB_Pansharpened_50cm.tif


array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.62097927, 0.68959577, 0.70716632, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])